In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Reduce the embeddings from 1024 to 10d and 2d via UMAP

In [ ]:
!pip install umap-learn -q

In [ ]:
# 1. SILENCE CONSOLE NOISE
import logging , warnings
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import umap
import pandas as pd
from datetime import datetime

# 1. Load your pre-computed embeddings
embeddings = np.load("/kaggle/input/datasets/sohamgangulycoder/cleaned-bge-m3-embeddings/bgm_m3_embeddings_after_data_cleaning.npy")

# 2. Dimensionality Reduction to 10D 
# We use 10D for clustering (better than 2D for math) 
# We will use 2D later JUST for the visualization galaxy
print("🚀 Reducing dimensions for clustering...")
reducer_clustering = umap.UMAP( 
    min_dist=0.1,
    n_neighbors=15, 
    n_components=10, # Higher dim for better FCM accuracy
    metric='cosine', 
    random_state=42
)
u_embeddings_clustering = reducer_clustering.fit_transform(embeddings)

# 3. Dimensionality Reduction to 2D (Specifically for Visualization #1)
print("🌌 Reducing dimensions for the Galaxy Map...")
reducer_viz = umap.UMAP(
    min_dist=0.1,
    n_neighbors=15, 
    n_components=2, 
    metric='cosine', 
    random_state=42
)
u_embeddings_viz = reducer_viz.fit_transform(embeddings)

print(f"✅ Success! Shapes: Clustering({u_embeddings_clustering.shape}), Viz({u_embeddings_viz.shape})")

In [ ]:
import json
import pandas as pd

# Load the full metadata and full text
data_list = []
input_file = "/kaggle/input/datasets/sohamgangulycoder/cleaned-merged-depression/merged_depression_final.jsonl"

print("📖 Reading full posts and timestamps...")
with open(input_file, 'r') as f:
    for line in f:
        item = json.loads(line)
        data_list.append({
            'created_utc': item.get('date'),
            'full_text': item.get('text')  # Keeping 100% of the content now
        })

df_meta = pd.DataFrame(data_list)

# Convert timestamps to datetime
try:
    df_meta['date'] = pd.to_datetime(df_meta['created_utc'], unit='s')
except Exception:
    df_meta['date'] = pd.to_datetime(df_meta['created_utc'])

# Sorting is CRITICAL for the StreamGraph and PELT later
df_meta = df_meta.sort_values('date').reset_index(drop=True)

print(f"✅ Data Synchronized.")
print(f"📊 Total Posts: {len(df_meta)}")
print(f"📅 Timeline: {df_meta['date'].min().year} to {df_meta['date'].max().year}")

In [ ]:
df_meta

In [ ]:
!pip install bertopic

In [ ]:
import json
import numpy as np
import pandas as pd
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN

# --- STEP 2: LOAD YOUR 10D EMBEDDINGS ---
docs = []
print("Reading JSONL...")
with open("/kaggle/input/datasets/sohamgangulycoder/cleaned-merged-depression/merged_depression_final.jsonl", 'r') as f:
    for line in f:
        data = json.loads(line)
        docs.append(data['text'])
print(f"Loaded {len(docs)} documents")

print(f"Embeddings shape: {u_embeddings_clustering.shape}")  # should be (17000+, 10)

# --- STEP 3: HDBSCAN — min_cluster_size=20 is your "20 posts" rule ---
hdbscan_model = HDBSCAN(
    min_cluster_size=50,    # a topic needs AT LEAST 20 posts to exist
    min_samples=15,          # how conservative outlier detection is — lower = more topics
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# --- STEP 4: STOP WORDS (Hindi + English function words + Reddit noise) ---
stop_words = list({
    # English function words
    "i","me","my","myself","we","our","you","your","he","him","his","she",
    "her","they","them","what","which","who","this","that","these","those",
    "am","is","are","was","were","be","been","being","have","has","had",
    "do","does","did","will","would","could","should","may","might","shall",
    "can","to","of","in","for","on","with","at","by","from","as","into",
    "through","about","against","between","but","if","or","because","until",
    "while","so","than","too","very","just","and","the","a","an","not","no",
    "nor","yet","both","either","each","more","most","other","such","own",
    "same","then","here","there","when","where","why","how","all","any",
    "few","also","even","really","want","wanted","know","think","thought",
    "get","got","like","still","well","going","go","say","said","one","make",
    "made","take","need","never","always","please","people","time","things",
    "thing","day","days","year","years","way","now","back","much","actually",
    "literally","something","anything","everything","nothing","im","dont",
    "ive","its","doesnt","cant","wont","didnt","isnt","wasnt","wouldnt",
    "couldnt","shouldnt","ill","id","iam","hes","shes","theyre","weve",
    # Depression noise — present in ALL posts equally (not discriminative)
    "feel","feeling","feelings","felt","depressed","depression","anxiety",
    "sad","help","life","advice","anyone","someone","everyone","nobody",
    # Reddit artifacts
    "removed","deleted","edit","update","tldr","post","comment","reddit",
    "upvote","karma","throwaway","account","username","op","oc",
    # Hindi function words (stop these so actual Hindi cause-words survive)
    "hai","ki","se","bhi","nahi","hu","ke","aur","nhi","ho","ko","ka","ne",
    "kya","koi","mujhe","mera","mere","ek","toh","par","jo","yeh","woh",
    "hain","tha","thi","the","hoga","hogi","kar","karna","karta","karti",
    "mein","main","hum","aap","tum","unhe","unka","unki","unse","inhe",
    "inka","inki","inse","yahan","wahan","ab","phir","fir","lekin","magar",
    "matlab","bas","sirf","bohot","bahut","thoda","kaafi","bilkul","haan",
    "nahi","nahin","na","baat","log","sab","kuch","koi","kaise","kyun",
    "kyunki","isliye","isiliye","tab","jab","agar","toh","to",
})

vectorizer_model = CountVectorizer(
    stop_words=stop_words,
    min_df=5,           # word must appear in at least 5 docs to count
    max_df=0.6,         # ignore words present in >40% of docs (too generic)
    ngram_range=(1, 2)  # bigrams like "financial_stress", "arranged_marriage"
)

# --- STEP 5: REPRESENTATION — MMR gives diverse, non-repetitive topic keywords ---
representation_model = MaximalMarginalRelevance(diversity=0.4)

# --- STEP 6: BUILD BERTOPIC ---
# IMPORTANT: pass umap_model=None since embeddings are already reduced to 10D
# BERTopic will skip its internal UMAP step and use your embeddings directly
topic_model = BERTopic(
    umap_model=None,                    # skip — you already have 10D embeddings
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    representation_model=representation_model,
    top_n_words=12,
    nr_topics=50,
    verbose=True
)

# --- STEP 7: FIT ---
print("Fitting BERTopic on 10D embeddings...")
topics, probs = topic_model.fit_transform(docs, u_embeddings_clustering)

# --- STEP 8: INSPECT ---
topic_info = topic_model.get_topic_info()

In [ ]:
topic_info.head(5)

In [ ]:
import plotly.express as px

# Create a DF for a quick look
df_viz = pd.DataFrame({
    'x': u_embeddings_viz[:, 0],
    'y': u_embeddings_viz[:, 1],
    'topic': topics,
    'text': [d[:100] for d in docs]
})

# Filter out outliers (-1) just to see the 'bright suns'
fig = px.scatter(df_viz[df_viz.topic != -1], x='x', y='y', color='topic', 
                 hover_data=['text'], title="Preliminary Galaxy (Hard Clusters)")
fig.show()

In [ ]:
len(topics)

In [ ]:
# 1. Get the centroids from the fitted topic model
# These are in the same 10-D space as u_embeddings_clustering
all_topic_embeddings = topic_model.topic_embeddings_ 

# 2. Get the list of actual topic IDs (excluding -1)
# BERTopic internal indexing: all_topic_embeddings[0] is usually Topic -1
# We only want the valid topics
valid_topic_indices = [topic + 1 for topic in topic_model.get_topics().keys() if topic != -1]
initial_centers = all_topic_embeddings[valid_topic_indices]

n_clusters = len(initial_centers)
print(f"🎯 Extracted {n_clusters} Topic Centers for FCM Warm-Start.")

In [ ]:
initial_centers.shape ## its 0 to 48 excluding -1

In [ ]:
from scipy.spatial.distance import cdist

# 1. Calculate Euclidean distance from every post (17k) to every center (n_clusters)
# Input: (17100, 10) and (n_clusters, 10) -> Output: (17100, n_clusters)
distances = cdist(u_embeddings_clustering, initial_centers, metric='euclidean')

# 2. Convert distances to memberships (Inverse distance logic)
# Closer to a center = Higher membership
u0 = 1.0 / (distances + 1e-8) 
u0 = u0 / u0.sum(axis=1)[:, None] # Normalize so each row sums to 1.0

# FCM expects (n_clusters, n_samples)
u0 = u0.T 
print(f"⚡ Initial Membership Matrix (u0) prepared: {u0.shape}")

In [ ]:
!pip install scikit-fuzzy -q


In [ ]:
# Run FCM with the Warm-Start matrix
import skfuzzy as fuzzy
cntr, u, u0_final, d, jm, p, fpc = fuzzy.cluster.cmeans(
    u_embeddings_clustering.T, 
    c=n_clusters, 
    m=1.2, 
    error=0.005, 
    maxiter=1000, 
    init=u0 # This is the magic line
)

print(f"✅ Guided FCM Complete. FPC Score: {fpc:.4f}")

In [ ]:
len(cntr),cntr.shape

### Mapping the Math to the Ledger

In [ ]:
# 1. Add the 2D Galaxy coordinates (from Step 1)
df_meta['x'] = u_embeddings_viz[:, 0]
df_meta['y'] = u_embeddings_viz[:, 1]

# 2. Add the Fuzzy Membership probabilities (u matrix)
# Each row in 'u' is a cluster, each column is a post
for i in range(n_clusters):
    df_meta[f'cluster_{i}_prob'] = u[i]

# 3. Identify the 'Primary Cluster' (highest probability)
# Useful for quick filtering and hard-labeling
import numpy as np
df_meta['primary_cluster'] = np.argmax(u, axis=0)

# 4. Calculate 'Fuzziness' (Entropy of the distribution)
# High value = Hybrid post | Low value = Pure stressor
df_meta['fuzziness'] = 1 - np.max(u, axis=0)

print("✅ Master Ledger updated with Fuzzy results.")
print(df_meta.columns)

In [ ]:
df_meta

## 1. The "Stability Test" (Centroid Drift)
## Before you spend any time re-labeling, you can mathematically check if the clusters stayed true to their original BERTopic identities. We use Cosine Similarity to compare the "Before" (BERTopic) and "After" (FCM) centers.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

for i in range(n_clusters):
    # initial_centers[i] is the BERTopic center
    # cntr[i] is the new Fuzzy C-Means center
    sim = cosine_similarity(initial_centers[i].reshape(1, -1), 
                            cntr[i].reshape(1, -1))[0][0]
    
    print(f"Cluster {i} Stability: {sim:.4f}")

#### The 1:1 Mapping is Locked: You can say with 100% mathematical certainty that Topic $X$ from BERTopic is Cluster $X$ in Fuzzy C-Means. You don't have to guess or cross-reference.No "Drift" to Explain: This makes your thesis defense much easier. You can state that the HDBSCAN (density-based) discovery was so robust that the Fuzzy C-Means (distance-based) optimization didn't need to adjust the coordinates.The Fuzzy c-TF-IDF is still the "Special Sauce": Even though the centers didn't move, the Fuzzy c-TF-IDF is still worth running.BERTopic's keywords only care about the "core" members of the topic.Fuzzy c-TF-IDF keywords account for the entire dataset, weighting words by how much a post belongs to that cluster. This will give you more "nuanced" markers that capture how these stressors bleed into each other.

### Step 1: Generate the "Meaningful Markers" (Zero-Shot)


In [ ]:
! pip install -U huggingface_hub

In [ ]:
# It is best practice to store this in Kaggle Secrets (Add-ons -> Secrets)
# and access it via: from kaggle_secrets import UserSecretsClient

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hugging_face_token")
hf_token = secret_value_0
login(token=hf_token)

In [ ]:
# Force-upgrade everything together to ensure version alignment
!pip install transformers accelerate huggingface_hub bitsandbytes -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

# 1. Clear old memory
gc.collect()
torch.cuda.empty_cache()

# 2. Load Gemma-2-2B-IT (Native Kaggle Path)
model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16 # No quantization needed!
)

print("💎 Gemma-2-2B is ready. No complex drivers required.")

In [ ]:
# # 1. Use your already defined vectorizer_model to transform the documents
# # This creates the (17398, V) matrix where V is your vocabulary size
# print("🔄 Transforming documents into the Bag-of-Words matrix...")
# word_counts = vectorizer_model.fit_transform(docs)

# # 2. Extract the words (and bigrams) from the model
# words = vectorizer_model.get_feature_names_out()

# print(f"✅ Matrix built! Shape: {word_counts.shape}")
# print(f"✅ Vocabulary contains {len(words)} terms (including bigrams).")

In [ ]:
gemma_prompt = """<start_of_turn>user
[DOCUMENTS]
Keywords: [KEYWORDS]

You are a sociological classifier. Summarize the core cause of distress in these documents into exactly ONE umbrella term (1-3 words).
Strict Rules:
1. No internal thoughts.
2. No prefix like "Umbrella Term:".
3. No bolding.
4. Output ONLY the term.

Umbrella Term:<end_of_turn>
<start_of_turn>model
"""

In [ ]:
from bertopic.representation import TextGeneration

# 1. Re-initialize with the missing parameters
gemma_representation = TextGeneration(
    model=model_id,            # Use the actual model object you loaded
    tokenizer=tokenizer,    # Use the tokenizer object
    prompt=gemma_prompt,
    nr_docs=5,              # How many docs to show Gemma (we'll use 5)
    doc_length=200,         # Max tokens per document to avoid context overflow
    pipeline_kwargs={"max_new_tokens": 150} 
)

# 2. Add to representation dictionary
representation_model = {"Umbrella_Marker": gemma_representation}

# 3. Update the topics
print("🧠 Gemma is synthesizing Umbrella Markers (5 docs per cluster)...")
topic_model.update_topics(docs, representation_model=representation_model)

# 4. View results
topic_info = topic_model.get_topic_info()
print(topic_info[["Topic", "Count", "Umbrella_Marker"]].head())

In [ ]:
topic_info['Umbrella_Marker']

In [ ]:
import re

def clean_marker(text):
    if isinstance(text, list): text = text[0] # Handle the list wrapper
    
    # 1. Remove "thought" blocks and reasoning
    text = re.split(r'thought|Based on|The documents|This cluster', text, flags=re.IGNORECASE)[0]
    
    # 2. Remove specific prefixes and formatting
    text = re.sub(r'Umbrella Term[:\s]*', '', text, flags=re.IGNORECASE)
    text = text.replace('**', '').replace('[', '').replace(']', '').strip()
    
    # 3. Clean up trailing commas/whitespace
    text = text.split(',')[0].strip()
    
    # 4. Final guardrail: If it's too long, it's probably a hallucination
    if len(text.split()) > 5:
        return "Complex Societal Stress"
    return text if text else "I am not sure"

# Apply the cleaning to your topic_info
topic_info['Umbrella_Marker_Clean'] = topic_info['Umbrella_Marker'].apply(clean_marker)

# Check the results now
print(topic_info[['Topic', 'Umbrella_Marker_Clean']].head(10))

In [ ]:
topic_info.iloc[:,-1]

In [ ]:
topic_info

In [ ]:
df_meta

In [ ]:
# 1. Create a clean dictionary mapping Topic ID to the Marker string
# This handles the -1 offset automatically
marker_dict = dict(zip(topic_info.Topic, topic_info.Umbrella_Marker_Clean))

# 2. Map this to your master ledger
# primary_cluster 0 will now correctly show 'Family Trauma'
df_meta['umbrella_marker'] = df_meta['primary_cluster'].map(marker_dict)

# 3. Handle the 'Family Abuse' (-1) as background noise
df_meta['umbrella_marker'] = df_meta['umbrella_marker'].fillna("General Distress/Outliers")

In [ ]:
df_meta

In [ ]:
# Mapping the CLEANED markers to Super-Clusters
super_map = {
    "Family Trauma": "Family & Domestic", "Family Abuse": "Family & Domestic", 
    "Emotional Neglect": "Family & Domestic", "Family Pressure": "Family & Domestic", 
    "Parental Pressure": "Family & Domestic", "Domestic Abuse": "Family & Domestic",
    "Parental conflict": "Family & Domestic", "Family Conflict": "Family & Domestic",
    "Family Dysfunction": "Family & Domestic", "Abuse and Trauma": "Family & Domestic",
    "Sexual Trauma": "Family & Domestic",
    
    "Existential crisis.": "Existential & Mood", "Depression": "Existential & Mood",
    "Existential Crisis": "Existential & Mood", "Mental health crisis": "Existential & Mood",
    "Mental Health Struggles": "Existential & Mood", "Emotional Exhaustion": "Existential & Mood",
    "Existential despair": "Existential & Mood", "Trauma": "Existential & Mood",
    
    "Social Isolation": "Social Isolation", "Loneliness and grief": "Social Isolation",
    "Social insecurity": "Social Isolation", "Emotional pain and isolation": "Social Isolation",
    "Social Anxiety and Identity": "Social Isolation", "Social isolation": "Social Isolation",
    
    "Betrayal and heartbreak": "Relationships", "Long distance relationship": "Relationships",
    "Relationship heartbreak": "Relationships", "family relationship issues": "Relationships",
    
    "Academic Pressure": "Academic & Career", "Work and Life Struggles": "Academic & Career",
    
    "Societal Corruption": "Societal & Misc", "Unrealistic expectations": "Societal & Misc",
    "Body image issues": "Societal & Misc", "Cultural stigma": "Societal & Misc",
    "Poor customer service": "Societal & Misc"
}

df_meta['super_cluster'] = df_meta['umbrella_marker'].map(super_map).fillna("General Distress/Outliers")

In [ ]:
df_meta

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_meta_final, 
    x='x', y='y',
    color='super_cluster',
    hover_data={'full_text': True, 'umbrella_marker': True, 'super_cluster': True, 'fuzziness': True},
    title="<b>Visualization 1: The Sociological Galaxy of Indian Distress</b>",
    color_discrete_sequence=px.colors.qualitative.Prism, 
    template="plotly_dark"
)

# Customizing for a "Galaxy" feel
fig.update_traces(
    marker=dict(size=2.5, opacity=0.4), # Small dots to see density clusters
    selector=dict(mode='markers')
)

fig.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    legend=dict(title="<b>Sociological Stressor Type</b>", font=dict(size=12))
)

fig.show()

In [ ]:
import plotly.express as px
import numpy as np

# Create a 'Purity' score for better visualization (Inverse of Fuzziness)
# We want the most certain posts to be the biggest stars
df_meta_final['purity'] = 1 - df_meta['fuzziness']

fig = px.scatter(
    df_meta, 
    x='x', y='y',
    color='super_cluster',
    size='purity',            # <--- This makes 'Pure' posts larger
    size_max=8,               # Keeps the stars from getting too big
    hover_data={
        'full_text': True, 
        'umbrella_marker': True, 
        'fuzziness': ':.4f',  # Show 4 decimal places
        'purity': False,
        'x': False, 
        'y': False
    },
    title="<b>Visualization 1: The Fuzzy Galaxy of Indian Distress</b>",
    color_discrete_sequence=px.colors.qualitative.Prism, 
    template="plotly_dark"
)

fig.update_traces(marker=dict(opacity=0.6)) # Slight transparency shows the density

fig.update_layout(
    plot_bgcolor='black',
    xaxis=dict(showgrid=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, showticklabels=False, title="")
)

fig.show()

In [ ]:
import pandas as pd

# 1. Create a reverse mapping: Topic ID -> Super Cluster Name
# We use the marker_dict we created earlier + the super_map taxonomy
topic_to_super = {
    tid: super_map[marker] 
    for tid, marker in marker_dict.items() 
    if marker in super_map
}

# 2. Sum the Probabilities
print("🧬 Engineering Super-Cluster Features...")
unique_super_names = set(super_map.values())

for sc_name in unique_super_names:
    # Identify all cluster_X_prob columns that belong to this Super Cluster
    cols_to_sum = [
        f"cluster_{tid}_prob" 
        for tid, sc in topic_to_super.items() 
        if sc == sc_name
    ]
    
    # Safety check: Only sum columns that actually exist in df_meta
    existing_cols = [c for c in cols_to_sum if c in df_meta.columns]
    
    if existing_cols:
        column_name = f"prob_{sc_name.replace(' ', '_').lower()}"
        df_meta[column_name] = df_meta[existing_cols].sum(axis=1)

# 3. Reduce: Drop the original 49 individual probability columns
# This keeps your CSV lean for Streamlit/Power BI
cols_to_drop = [c for c in df_meta.columns if c.startswith('cluster_') and c.endswith('_prob')]
df_meta_final = df_meta.drop(columns=cols_to_drop)

# 4. Export the clean version
df_meta_final.to_csv("distress_final_reduced.csv", index=False)
print(f"✅ Success! Columns reduced from {len(df_meta.columns)} to {len(df_meta_final.columns)}")

In [ ]:
df_meta_final

In [ ]:
df_meta_final.columns

### Step 2: Semantic Color BlendingTo show "Hybrid Stressors," we don't just give each cluster a solid color. We blend the RGB values based on the fuzzy membership. A post that is $50\%$ Red (Academic) and $50\%$ Blue (Career) will appear Purple.

### Step 1: Feature Extraction (The "Marker" Creator)

In [ ]:
# Force-upgrade everything together to ensure version alignment
!pip install transformers accelerate huggingface_hub bitsandbytes -q

In [ ]:
import logging , warnings